# 📊 Base de Dados Dinâmica — Solução Completa (Tudo em Um)

Este notebook contém toda a lógica para:
1. **Criar tabelas** para qualquer ano automaticamente
2. **Inserir dados** sem duplicatas
3. **Inspecionar a BD** com ferramentas interativas

## ✨ Vantagens
- ✅ Detecta automaticamente quais anos estão nos ficheiros
- ✅ Cria tabelas para TODOS os anos (não apenas 2025/2026)
- ✅ Insere dados em qualquer ano sem editar código
- ✅ Protege contra duplicatas automaticamente
- ✅ Seguro para correr múltiplas vezes

## 🚀 Modo de Usar
Executa as células **por ordem**, uma a uma (Cell → Run All também funciona)


## PASSO 1 — Importações e Configuração

In [1]:
# ==============================================================================
# IMPORTAÇÕES
# ==============================================================================
import platform
import sqlite3
import os
import glob
import xml.etree.ElementTree as ET
from datetime import datetime
from collections import defaultdict

print("✓ Bibliotecas importadas com sucesso")

✓ Bibliotecas importadas com sucesso


In [2]:
# ==============================================================================
# CONFIGURAÇÃO DE CAMINHOS
# ==============================================================================

if platform.system() == 'Windows':
    DB_PATH = r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db"
    PASTA_FICHEIROS = r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27"
elif platform.system() == 'Darwin':
    DB_PATH = "/Volumes/RR/DB/inform_27.db"
    PASTA_FICHEIROS = "/Volumes/RR/DB/inform_27"
else:
    DB_PATH = "inform_27.db"
    PASTA_FICHEIROS = "inform_27"

print(f"DB_PATH: {DB_PATH}")
print(f"PASTA_FICHEIROS: {PASTA_FICHEIROS}")
print()

# Verificar/Criar pasta se não existir
pasta = os.path.dirname(DB_PATH)
if pasta and not os.path.exists(pasta):
    os.makedirs(pasta, exist_ok=True)
    print(f"Pasta criada: {pasta}")

# Criar BD
con = sqlite3.connect(DB_PATH)
con.close()
print(f"✓ Base de dados criada/aberta em: {DB_PATH}")

DB_PATH: C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db
PASTA_FICHEIROS: C:\Users\LISARR\Documents\python\01.Financeiro\inform_27

✓ Base de dados criada/aberta em: C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db


In [3]:
# ==============================================================================
# DEFINIÇÕES GLOBAIS
# ==============================================================================

COLUNAS_FICHEIRO = [
    "PROPIETARIO", "TRAYECTO", "TRANSPORTISTA", "TRACTORA", "REMOLQUE",
    "INGRESODT", "COSTEDT", "RENTADT", "PALETSDT", "PESO_BRUTO",
    "CODEUT", "ESTADO_UT", "RANGO_UT", "FCARGA", "ACTIVIDAD",
    "CODEDT", "ESTADO_DT", "REFERENCIA", "CODACT", "LOCORIGEN",
    "PROV_ORIGEN", "PAISORIGEN", "CPOSTAL", "LOCDESTINO", "PROV_DESTINO",
    "PAISDESTINO", "CPOSTAD", "KM", "FENTREGA", "ORIGEN",
    "ENTREGAR", "PROV_ENTREGAR", "PAISENTREGAR", "DESTINO", "PALETS",
    "PREFAC", "RUTA", "COBROREAL", "GESTION", "DEPART",
    "USCODE", "USUARIO", "TIPOCLIENTE", "TIPOFLUJO", "WMSCODRGT",
    "LOCCAR", "LUGARCARGA", "LOCDES", "LUGARDESCARGA", "TEMP_MERC_PED",
    "TIPOPALETA", "CAMION_TIPO", "CAMION_CAPACIDAD", "TIPO_COMBUSTIBLE", "KMREALES",
    "ALBARAN",
]

COLUNA_ORIGEM = "ficheiro_origem"
CAMPO_DATA = "FENTREGA"

print(f"✓ {len(COLUNAS_FICHEIRO)} colunas definidas")
print(f"✓ Campo de data: {CAMPO_DATA}")
print(f"✓ Coluna de origem: {COLUNA_ORIGEM}")

✓ 56 colunas definidas
✓ Campo de data: FENTREGA
✓ Coluna de origem: ficheiro_origem


## PASSO 2 — Funções Auxiliares

In [4]:
# ==============================================================================
# FUNÇÃO: Ler ficheiros Excel XML
# ==============================================================================

NS = {"ss": "urn:schemas-microsoft-com:office:spreadsheet"}
SS_INDEX = "{urn:schemas-microsoft-com:office:spreadsheet}Index"

def ler_linhas_xml_spreadsheet(caminho_ficheiro):
    """
    Lê um ficheiro no formato 'Excel XML Spreadsheet 2003' e devolve
    (cabecalho, lista_de_linhas).
    """
    try:
        tree = ET.parse(caminho_ficheiro)
        root = tree.getroot()

        worksheet = root.find("ss:Worksheet", NS)
        if worksheet is None:
            return None, None

        tabela = worksheet.find("ss:Table", NS)
        if tabela is None:
            return None, None

        linhas_xml = tabela.findall("ss:Row", NS)
        if not linhas_xml:
            return [], []

        def extrair_linha(linha_xml):
            valores = []
            proximo_indice = 1
            for cell in linha_xml.findall("ss:Cell", NS):
                idx = cell.get(SS_INDEX)
                idx = int(idx) if idx is not None else proximo_indice
                while len(valores) < idx - 1:
                    valores.append(None)
                data_el = cell.find("ss:Data", NS)
                valor = data_el.text if data_el is not None else None
                valores.append(valor)
                proximo_indice = idx + 1
            return valores

        cabecalho = extrair_linha(linhas_xml[0])
        cabecalho = [c.strip() if c else f"COLUNA_{i+1}" for i, c in enumerate(cabecalho)]

        linhas = []
        n_colunas = len(cabecalho)
        for linha_xml in linhas_xml[1:]:
            valores = extrair_linha(linha_xml)
            if len(valores) < n_colunas:
                valores += [None] * (n_colunas - len(valores))
            elif len(valores) > n_colunas:
                valores = valores[:n_colunas]
            linhas.append(valores)

        return cabecalho, linhas
    except Exception as e:
        print(f"    [ERRO ao ler] {e}")
        return None, None

print("✓ Função ler_linhas_xml_spreadsheet() pronta")

✓ Função ler_linhas_xml_spreadsheet() pronta


In [5]:
# ==============================================================================
# FUNÇÃO: Listar ficheiros Excel
# ==============================================================================

def listar_ficheiros_excel(pasta):
    """Procura todos os ficheiros .xls na pasta (incluindo subpastas)."""
    encontrados = glob.glob(os.path.join(pasta, "**", "*.xls"), recursive=True)
    vistos = set()
    ficheiros = []
    for caminho in encontrados:
        chave = os.path.normcase(os.path.abspath(caminho))
        if chave not in vistos:
            vistos.add(chave)
            ficheiros.append(caminho)
    return sorted(ficheiros)

print("✓ Função listar_ficheiros_excel() pronta")

✓ Função listar_ficheiros_excel() pronta


In [6]:
# ==============================================================================
# FUNÇÃO: Detectar anos nos ficheiros
# ==============================================================================

def detectar_anos_nos_ficheiros(pasta):
    """
    Procura todos os ficheiros .xls na pasta e detecta quais são os anos
    presentes nos dados (através do campo FENTREGA).
    Devolve um conjunto (set) com os anos encontrados.
    """
    ficheiros = listar_ficheiros_excel(pasta)
    anos = set()
    
    idx_data = COLUNAS_FICHEIRO.index(CAMPO_DATA)
    
    for caminho in ficheiros:
        cabecalho, linhas = ler_linhas_xml_spreadsheet(caminho)
        if cabecalho is None or CAMPO_DATA not in cabecalho:
            continue
        
        for linha in linhas:
            if linha and idx_data < len(linha):
                valor_data = linha[idx_data]
                if valor_data and str(valor_data).strip()[:4].isdigit():
                    ano = str(valor_data).strip()[:4]
                    anos.add(ano)
    
    return sorted(anos)

print("✓ Função detectar_anos_nos_ficheiros() pronta")

✓ Função detectar_anos_nos_ficheiros() pronta


In [7]:
# ==============================================================================
# FUNÇÃO: Detectar anos na BD
# ==============================================================================

def detectar_anos_na_bd(db_path):
    """
    Detecta quais são as tabelas de anos presentes na BD.
    Devolve uma lista de anos (ex: ['2023', '2024', '2025', '2026'])
    """
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tabelas = [nome for (nome,) in cur.fetchall()]
    
    con.close()
    
    anos = []
    for tabela in tabelas:
        if tabela.startswith("inform_27_"):
            ano = tabela.replace("inform_27_", "")
            if ano.isdigit() and len(ano) == 4:
                anos.append(ano)
    
    return sorted(anos)

print("✓ Função detectar_anos_na_bd() pronta")

✓ Função detectar_anos_na_bd() pronta


## PASSO 3 — Criar Tabelas Dinamicamente

In [8]:
# ==============================================================================
# DETECTAR ANOS NOS FICHEIROS
# ==============================================================================

print("A detectar anos nos ficheiros...")
anos_detectados = detectar_anos_nos_ficheiros(PASTA_FICHEIROS)

print()
if anos_detectados:
    print(f"✓ Anos detectados: {', '.join(anos_detectados)}")
else:
    print("[AVISO] Nenhum ano detectado. Verifica se:")
    print("  - Os ficheiros .xls existem em:", PASTA_FICHEIROS)
    print("  - Têm a coluna FENTREGA")
    print("  - FENTREGA tem valores no formato AAAAMMDD")

A detectar anos nos ficheiros...

✓ Anos detectados: 0000, 2026


In [9]:
# ==============================================================================
# CRIAR TABELAS PARA CADA ANO DETECTADO
# ==============================================================================

if not anos_detectados:
    print("[ERRO] Sem anos para processar. Verifica a pasta de ficheiros.")
else:
    colunas_sql = ",\n    ".join(f'"{c}" TEXT' for c in COLUNAS_FICHEIRO)

    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()

    print(f"A criar {len(anos_detectados)} tabela(s)...")
    print()
    
    for ano in anos_detectados:
        tabela_nome = f"inform_27_{ano}"
        cur.execute(f'''
            CREATE TABLE IF NOT EXISTS {tabela_nome} (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                {colunas_sql},
                "{COLUNA_ORIGEM}" TEXT
            )
        ''')
        print(f"  ✓ Tabela {tabela_nome} criada (ou já existia).")

    con.commit()
    con.close()
    
    print()
    print("✓ Todas as tabelas prontas!")

A criar 2 tabela(s)...

  ✓ Tabela inform_27_0000 criada (ou já existia).
  ✓ Tabela inform_27_2026 criada (ou já existia).

✓ Todas as tabelas prontas!


In [10]:
# ==============================================================================
# CONFIRMAR: Listar tabelas criadas
# ==============================================================================

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tabelas_bd = [nome for (nome,) in cur.fetchall()]

print("Tabelas na BD:")
for tabela in tabelas_bd:
    print(f"  - {tabela}")

print()
for ano in anos_detectados:
    tabela_nome = f"inform_27_{ano}"
    cur.execute(f"PRAGMA table_info({tabela_nome})")
    colunas = cur.fetchall()
    print(f"Colunas de {tabela_nome}: {len(colunas)}")

con.close()
print()
print("✓ Estrutura confirmada!")

Tabelas na BD:
  - cargas_2023
  - cargas_2024
  - cargas_2025
  - cargas_2026
  - coordenadas
  - entregas_2025
  - entregas_2026
  - inform_26_2024
  - inform_26_2025
  - inform_26_2026
  - inform_27_0000
  - inform_27_2023
  - inform_27_2024
  - inform_27_2025
  - inform_27_2026
  - km_diario_2023
  - km_diario_2024
  - km_diario_2025
  - km_diario_2026
  - sqlite_sequence

Colunas de inform_27_0000: 58
Colunas de inform_27_2026: 58

✓ Estrutura confirmada!


## PASSO 4 — Inserir Dados Dinamicamente

In [11]:
# ==============================================================================
# PREPARAR INSTRUÇÕES SQL (INSERT OR IGNORE) PARA CADA ANO
# ==============================================================================

cols_sql = ", ".join(f'"{c}"' for c in COLUNAS_FICHEIRO)
placeholders = ", ".join(["?"] * len(COLUNAS_FICHEIRO))
idx_data = COLUNAS_FICHEIRO.index(CAMPO_DATA)

# Gerar SQL para cada ano
sql_por_ano = {}
for ano in anos_detectados:
    tabela_nome = f"inform_27_{ano}"
    sql = f'INSERT OR IGNORE INTO {tabela_nome} ({cols_sql}, "{COLUNA_ORIGEM}") VALUES ({placeholders}, ?)'
    sql_por_ano[ano] = sql

print(f"✓ Instruções SQL geradas para {len(sql_por_ano)} ano(s):")
for ano in sorted(sql_por_ano.keys()):
    print(f"  - inform_27_{ano}")

✓ Instruções SQL geradas para 2 ano(s):
  - inform_27_0000
  - inform_27_2026


In [12]:
# ==============================================================================
# CRIAR ÍNDICES ÚNICOS (evita duplicatas)
# ==============================================================================

print("A criar índices únicos em CODEDT...")
print()

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

for ano in anos_detectados:
    tabela_nome = f"inform_27_{ano}"
    indice_nome = f"idx_{tabela_nome}_codedt"
    try:
        cur.execute(f'''
            CREATE UNIQUE INDEX IF NOT EXISTS {indice_nome}
            ON {tabela_nome}("CODEDT")
        ''')
        print(f"  ✓ Índice {indice_nome} pronto.")
    except sqlite3.OperationalError:
        print(f"  ✓ Índice {indice_nome} já existia.")

con.commit()
con.close()
print()
print("✓ Índices criados!")

A criar índices únicos em CODEDT...

  ✓ Índice idx_inform_27_0000_codedt pronto.
  ✓ Índice idx_inform_27_2026_codedt pronto.

✓ Índices criados!


In [13]:
# ==============================================================================
# LISTAR FICHEIROS A PROCESSAR
# ==============================================================================

ficheiros = listar_ficheiros_excel(PASTA_FICHEIROS)

print(f"Ficheiros .xls encontrados: {len(ficheiros)}")
print()
for i, f in enumerate(ficheiros[:20], 1):
    print(f"  {i}. {os.path.basename(f)}")

if len(ficheiros) > 20:
    print(f"  ... e mais {len(ficheiros) - 20} ficheiro(s)")

print()
if not ficheiros:
    print("[AVISO] Nenhum ficheiro .xls encontrado. Verifica:")
    print(f"  - PASTA_FICHEIROS: {PASTA_FICHEIROS}")
    print(f"  - Se os ficheiros têm extensão .xls")

Ficheiros .xls encontrados: 1

  1. SAL_DAT027 (4)_agosto.xls



In [14]:
# ==============================================================================
# LOOP PRINCIPAL: Ler e inserir dados
# ==============================================================================

if not ficheiros:
    print("[ERRO] Sem ficheiros para processar.")
else:
    print("=" * 70)
    print("A PROCESSAR FICHEIROS")
    print("=" * 70)
    print()

    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()

    # Dicionários para contar inserções
    totais_inseridos = {ano: 0 for ano in anos_detectados}
    totais_duplicados = 0
    totais_sem_data = 0

    for i, caminho in enumerate(ficheiros, start=1):
        nome_ficheiro = os.path.basename(caminho)
        print(f"[{i}/{len(ficheiros)}] A processar: {nome_ficheiro}")

        cabecalho, linhas = ler_linhas_xml_spreadsheet(caminho)
        if cabecalho is None:
            print(f"  [ERRO] Falha a ler o ficheiro. A saltar.")
            continue

        if CAMPO_DATA not in cabecalho:
            print(f"  [AVISO] Coluna '{CAMPO_DATA}' não encontrada. A saltar.")
            continue

        # Organizar linhas por ano
        batches_por_ano = {ano: [] for ano in anos_detectados}
        sem_data = 0

        for valores in linhas:
            # Ignorar linhas vazias
            if all(v is None or str(v).strip() == "" for v in valores):
                continue

            # Remapaer para ordem fixa
            mapa = dict(zip(cabecalho, valores))
            valores_ordenados = [mapa.get(c) for c in COLUNAS_FICHEIRO]

            # Extrair e validar data
            valor_data = valores_ordenados[idx_data]
            if not valor_data or not str(valor_data).strip()[:4].isdigit():
                sem_data += 1
                continue

            texto_data = str(valor_data).strip()
            ano = texto_data[:4]

            # Adicionar ao batch do ano correspondente
            if ano in batches_por_ano:
                batches_por_ano[ano].append(valores_ordenados + [nome_ficheiro])
            else:
                sem_data += 1

        # Inserir batches
        for ano in anos_detectados:
            batch = batches_por_ano[ano]
            if not batch:
                continue

            antes = con.total_changes
            cur.executemany(sql_por_ano[ano], batch)
            con.commit()
            inseridos = con.total_changes - antes
            totais_inseridos[ano] += inseridos

            # Contar duplicatas
            duplicados_neste_ano = len(batch) - inseridos
            totais_duplicados += duplicados_neste_ano

        totais_sem_data += sem_data

        # Resumo desta linha
        resumo = ", ".join(f"{totais_inseridos[a]} novo(s) em {a}" for a in sorted(anos_detectados))
        print(f"  -> {resumo}")
        print(f"     {totais_duplicados} duplicata(s), {totais_sem_data} sem data")
        print()

    con.close()

    # Resumo final
    print("=" * 70)
    print("RESUMO FINAL")
    print("=" * 70)
    print()
    print(f"Ficheiros processados: {len(ficheiros)}")
    print()
    for ano in sorted(anos_detectados):
        print(f"  Linhas novas inseridas em {ano}: {totais_inseridos[ano]:,}")
    print()
    print(f"Linhas já existentes (duplicadas por CODEDT): {totais_duplicados:,}")
    print(f"Linhas sem FENTREGA válido: {totais_sem_data:,}")
    print("=" * 70)
    print()
    print("✓ Inserção de dados concluída!")

A PROCESSAR FICHEIROS

[1/1] A processar: SAL_DAT027 (4)_agosto.xls
  -> 4 novo(s) em 0000, 54814 novo(s) em 2026
     10716 duplicata(s), 0 sem data

RESUMO FINAL

Ficheiros processados: 1

  Linhas novas inseridas em 0000: 4
  Linhas novas inseridas em 2026: 54,814

Linhas já existentes (duplicadas por CODEDT): 10,716
Linhas sem FENTREGA válido: 0

✓ Inserção de dados concluída!


## PASSO 5 — Inspecionar a Base de Dados

In [15]:
# ==============================================================================
# ESTATÍSTICAS POR TABELA/ANO
# ==============================================================================

print("=" * 70)
print("INSPEÇÃO DA BASE DE DADOS")
print("=" * 70)
print()

con = sqlite3.connect(DB_PATH)
con.row_factory = sqlite3.Row
cur = con.cursor()

# Detectar tabelas de dados
anos_na_bd = detectar_anos_na_bd(DB_PATH)

print(f"Anos com tabelas na BD: {', '.join(anos_na_bd)}")
print()

total_geral = 0
for ano in anos_na_bd:
    tabela = f"inform_27_{ano}"
    
    # Contar registos
    cur.execute(f"SELECT COUNT(*) FROM {tabela}")
    count = cur.fetchone()[0]
    total_geral += count
    
    # Contar ficheiros únicos
    cur.execute(f"SELECT COUNT(DISTINCT ficheiro_origem) FROM {tabela}")
    num_ficheiros = cur.fetchone()[0]
    
    # Data mais antiga e recente
    cur.execute(f"SELECT MIN(FENTREGA), MAX(FENTREGA) FROM {tabela}")
    row = cur.fetchone()
    data_min = row[0] if row[0] else "N/A"
    data_max = row[1] if row[1] else "N/A"
    
    # Contar CODEDTs únicos
    cur.execute(f"SELECT COUNT(DISTINCT CODEDT) FROM {tabela}")
    codedt_unicos = cur.fetchone()[0]
    
    print(f"📊 inform-27_{ano}:")
    print(f"   • Registos totais: {count:,}")
    print(f"   • CODEDT únicos: {codedt_unicos:,}")
    print(f"   • Ficheiros importados: {num_ficheiros}")
    print(f"   • Data mais antiga: {data_min}")
    print(f"   • Data mais recente: {data_max}")
    print()

print(f"📈 TOTAL DE REGISTOS NA BD: {total_geral:,}")
print()

con.close()

INSPEÇÃO DA BASE DE DADOS

Anos com tabelas na BD: 0000, 2023, 2024, 2025, 2026

📊 inform-27_0000:
   • Registos totais: 14
   • CODEDT únicos: 14
   • Ficheiros importados: 3
   • Data mais antiga: 00000000
   • Data mais recente: 00000000

📊 inform-27_2023:
   • Registos totais: 106,399
   • CODEDT únicos: 106,399
   • Ficheiros importados: 7
   • Data mais antiga: 20230516
   • Data mais recente: 20231231

📊 inform-27_2024:
   • Registos totais: 186,335
   • CODEDT únicos: 186,335
   • Ficheiros importados: 9
   • Data mais antiga: 20240102
   • Data mais recente: 20241231

📊 inform-27_2025:
   • Registos totais: 190,938
   • CODEDT únicos: 190,938
   • Ficheiros importados: 35
   • Data mais antiga: 20250102
   • Data mais recente: 20251231

📊 inform-27_2026:
   • Registos totais: 163,033
   • CODEDT únicos: 163,033
   • Ficheiros importados: 22
   • Data mais antiga: 20260102
   • Data mais recente: 20260810

📈 TOTAL DE REGISTOS NA BD: 646,719



In [16]:
# ==============================================================================
# FICHEIROS IMPORTADOS (POR TABELA)
# ==============================================================================

print("=" * 70)
print("FICHEIROS IMPORTADOS")
print("=" * 70)
print()

con = sqlite3.connect(DB_PATH)
cur = con.cursor()

for ano in anos_na_bd:
    tabela = f"inform_27_{ano}"
    cur.execute(f"SELECT DISTINCT ficheiro_origem, COUNT(*) FROM {tabela} "
                f"GROUP BY ficheiro_origem ORDER BY COUNT(*) DESC")
    ficheiros_result = cur.fetchall()
    
    if ficheiros_result:
        print(f"📁 inform_27_{ano}:")
        for ficheiro, count in ficheiros_result:
            print(f"   • {ficheiro}: {count:,} linhas")
    else:
        print(f"📁 inform_27_{ano}: (vazio)")
    print()

con.close()

FICHEIROS IMPORTADOS

📁 inform_27_0000:
   • SAL_DAT027 (27)-.xls: 6 linhas
   • SAL_DAT027 (4)_agosto.xls: 4 linhas
   • SAL_DAT027 (26)-.xls: 4 linhas

📁 inform_27_2023:
   • SAL_DAT027 (23)-.xls: 60,283 linhas
   • SAL_DAT027 (24)-.xls: 46,080 linhas
   • SAL_DAT027 (19)-.xls: 12 linhas
   • SAL_DAT027 (26)-.xls: 8 linhas
   • SAL_DAT027 (21)-.xls: 7 linhas
   • SAL_DAT027 (22)-.xls: 6 linhas
   • SAL_DAT027 (27)-.xls: 3 linhas

📁 inform_27_2024:
   • SAL_DAT027 (29)-.xls: 60,394 linhas
   • SAL_DAT027 (26)-.xls: 58,759 linhas
   • SAL_DAT027 (27)-.xls: 48,169 linhas
   • SAL_DAT027 (30)-.xls: 18,694 linhas
   • SAL_DAT027 (24)-.xls: 227 linhas
   • SAL_DAT027 (23)-.xls: 77 linhas
   • SAL_DAT027 (25)-.xls: 10 linhas
   • SAL_DAT027 (29).xls: 3 linhas
   • SAL_DAT027 (28)-.xls: 2 linhas

📁 inform_27_2025:
   • SAL_DAT027 (29).xls: 10,454 linhas
   • SAL_DAT027 (35).xls: 9,947 linhas
   • SAL_DAT027 (32).xls: 9,751 linhas
   • SAL_DAT027 (48).xls: 8,787 linhas
   • SAL_DAT027 (53).xl